In [0]:
!pip install statsforecast

In [0]:
import pyspark.sql.functions as F
from statsforecast import StatsForecast
import pandas as pd
import numpy as np
import time

from sklearn.metrics import mean_absolute_percentage_error
from statsforecast.models import (
    HoltWinters,
    WindowAverage,
    DynamicOptimizedTheta as DOT,
    AutoARIMA,
    AutoETS

)

In [0]:
joined_data_path = "/Volumes/workspace/m5/processed/joined_data"

In [0]:
data = pd.read_parquet(joined_data_path)
data.groupby('state_id',as_index=True).agg(record_count=('id','count'),unique_ids = ('id','nunique')).display()

In [0]:

data = pd.read_parquet(joined_data_path)
prod_cus_dim = data[['id','item_id','dept_id','cat_id','store_id','state_id']].drop_duplicates().rename(columns={'id':"unique_id"})

In [0]:
data = data[['id','yearweek','sales']].drop_duplicates()
data = data.sort_values(by=['id','yearweek'])
data["date"] = pd.to_datetime(data['yearweek'].astype(str) + "1", format="%G%V%u")
data = data[data['yearweek']!="201653"]
data.head()

In [0]:
df = data[['id','date','sales']].rename(columns = {'id':'unique_id','date':'ds','sales':'y'})
df.head()

In [0]:
df.agg(min_yw = ('ds','min'),max_yw=('ds','max')).display()

In [0]:
train_df = df.groupby('unique_id',as_index=False).apply(lambda df:df.iloc[:-20,:])
val_df = df.groupby('unique_id',as_index=False).apply(lambda df:df.iloc[-20:,:])

In [0]:
season_length = 52
stat_models = [
    DOT(season_length=season_length),
    # AutoARIMA(season_length=season_length),
    # AutoETS(season_length=season_length),
]
fallback_model = WindowAverage(window_size=52)

# HoltWinters(),
# Croston(),

In [0]:
train_df['unique_id'].nunique()

In [0]:
week_count = train_df.groupby('unique_id',as_index=False).agg(hist_length=('ds','nunique'))
week_count['inscope_for_statmodels'] = (week_count['hist_length']>=2*season_length).fillna(0).astype('int')

In [0]:
week_count['inscope_for_statmodels'].value_counts()

In [0]:
train_df_inscope = train_df.merge(week_count,on=['unique_id'],how='left')
train_df_inscope = train_df_inscope[train_df_inscope['inscope_for_statmodels']==1][['unique_id','ds','y']]

In [0]:
sf = StatsForecast(
    models = stat_models,
    freq="W-MON",
)

y_pred_dot = sf.forecast(df=train_df_inscope,h=20)

In [0]:
sf_arima = StatsForecast(
    models = [AutoARIMA(season_length=52)],
    freq="W-MON",
)

y_pred_arima = sf_arima.forecast(df=train_df_inscope,h=20)


In [0]:
sf_auto_ets = StatsForecast(
    models = [AutoETS(season_length=season_length)],
    freq="W-MON",
)

y_pred_hw = sf_auto_ets.forecast(df=train_df_inscope,h=20)

In [0]:
validation_df = y_pred_dot.merge(val_df,on=['unique_id','ds'],how='left').merge(prod_cus_dim,on=['unique_id'],how='left')
validation_df.head()

In [0]:
state_cat_df = validation_df.groupby(['state_id','cat_id'],as_index=False).agg(dot_pred = ('DynamicOptimizedTheta','sum'),
                                                                y_true = ('y','sum'),
                                                                item_count = ('unique_id','nunique'))

state_cat_df['dot_mape'] = (
    np.round(abs(state_cat_df['y_true'] - state_cat_df['dot_pred'])
    / state_cat_df['y_true'],2)
    * 100
)


state_cat_df.display()

In [0]:
sf.plot(df=df,
        forecasts_df=y_pred_dot,
        unique_ids=['FOODS_3_090_CA_3','FOODS_3_586_TX_2','FOODS_3_090_WI_3','FOODS_3_586_TX_3'],
        models = ["DynamicOptimizedTheta"])

In [0]:
sf.plot(df=df,
        forecasts_df=y_pred_dot,
        unique_ids = ['FOODS_3_086_WI_3','FOODS_3_234_WI_2','FOODS_3_030_TX_3','FOODS_3_086_TX_1'],
        models = ["DynamicOptimizedTheta"]
        )

In [0]:
sf.plot(df=df,
        forecasts_df=y_pred_dot,
        unique_ids = ['FOODS_3_422_WI_2','FOODS_3_059_WI_2','FOODS_3_540_CA_2','FOODS_3_800_TX_3'],
        models = ["DynamicOptimizedTheta"]
        )

In [0]:
sf.plot(df=df,
        forecasts_df=y_pred_dot,
        unique_ids = ['FOODS_2_360_WI_2','FOODS_2_013_WI_2','FOODS_3_587_TX_3','FOODS_3_587_WI_3'],
        models = ["DynamicOptimizedTheta"]
        )